# 链接预测工程：时序切分、负采样与按 source 排名

链接预测不是把现有边随机拆成 train/test 后做二分类。本 Notebook 使用虚构社交图事件流，严格按时间构造三个任务：用 day≤3 预测 day4–5、用 day≤5 验证 day6–7、用 day≤7 测试 day8–9。候选必须在当时已存在、同 tenant、非自环且尚未连接；特征只从预测时刻的历史图计算。

实现覆盖 random/hard/mixed negative、false negative 与开放/封闭世界、Common Neighbors/Jaccard/Adamic–Adar/Preferential Attachment、轻量分类 ranker、AUC/AP、按 source 的 Hits@K/MRR、cold start、validation 阈值、在线候选生成、增量图、权限和可观测。数据受控且很小，结果只验证合同。


## 1. 外部合同与时间语义

请求包含可信 `AuthContext`、source ID、预测时刻 `cutoff`、horizon、K 和模型版本。响应给出候选、分数、证据特征、快照水位和降级路径。`cutoff` 之后的边不得参与邻居、度数、候选或特征。tenant 过滤必须在候选枚举前完成；否则“被过滤掉的高分节点”本身也是结构泄漏。这里的整数 day 是固定 ISO 日期的简写，区间统一采用 `(cutoff, horizon_end]`。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations
import hashlib
import json
import math
import time

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RNG = np.random.default_rng(23)
FEATURE_ORDER = ("common_neighbors", "jaccard", "adamic_adar", "preferential_attachment", "degree_u", "degree_v", "same_component")

@dataclass(frozen=True)
class AuthContext:
    tenant: str
    scopes: frozenset[str]
    principal: str
    def require(self, scope: str):
        if scope not in self.scopes:
            raise PermissionError(scope)

def fingerprint(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:18]

def stable_id(*parts) -> str:
    return fingerprint([str(p) for p in parts])

def pair(u: str, v: str) -> tuple[str, str]:
    if u == v:
        raise ValueError("不允许 self-loop candidate")
    return tuple(sorted((u, v)))

FEATURE_SCHEMA_ID = fingerprint({"version": "link-features-v1", "feature_order": list(FEATURE_ORDER)})
MODEL_VERSION = "temporal-link-logreg-v1"
auth_a = AuthContext("tenant-a", frozenset({"graph:read", "graph:write", "model:predict"}), "alice")
assert pair("b", "a") == ("a", "b")
assert len(FEATURE_ORDER) == 7 and stable_id("a|b", "c") != stable_id("a", "b|c")


## 2. 受控事件流与稳定事件 ID

u10 在 day5 出现、u11 在 day7 出现，用于验证冷启动。边事件不可变，event ID 由 tenant、端点和首次发生日生成；重复投递不应产生第二条边。真实系统还需区分事件时间与摄取时间、处理撤销/删除，并以 watermark 管理乱序。


In [ ]:
NODES = [("tenant-a", f"u{i:02d}", 1 if i < 10 else (5 if i == 10 else 7)) for i in range(12)] + [("tenant-b", "b0", 1), ("tenant-b", "b1", 1)]
EDGE_SPECS = [
    (1, "u00", "u01"), (1, "u01", "u02"), (1, "u02", "u03"), (1, "u03", "u04"),
    (1, "u04", "u05"), (1, "u05", "u00"), (1, "u06", "u07"), (1, "u07", "u08"),
    (1, "u08", "u09"), (1, "u09", "u06"), (2, "u01", "u03"), (2, "u02", "u04"),
    (2, "u07", "u09"), (3, "u00", "u02"), (3, "u06", "u08"), (3, "u05", "u06"),
    # train labels: (3,5]
    (4, "u00", "u03"), (4, "u01", "u04"), (4, "u02", "u06"), (5, "u02", "u05"), (5, "u05", "u07"),
    # validation labels: (5,7]
    (6, "u00", "u04"), (6, "u03", "u05"), (6, "u06", "u10"), (7, "u07", "u10"), (7, "u04", "u06"),
    # test labels: (7,9]
    (8, "u01", "u05"), (8, "u08", "u10"), (8, "u00", "u11"),
    (9, "u04", "u07"), (9, "u09", "u11"), (9, "u03", "u06"),
]
EVENTS = []
for day, u, v in EDGE_SPECS:
    EVENTS.append({"event_id": stable_id("tenant-a", *pair(u, v), day), "tenant": "tenant-a", "u": u, "v": v, "day": day})
EVENTS.append({"event_id": stable_id("tenant-b", "b0", "b1", 1), "tenant": "tenant-b", "u": "b0", "v": "b1", "day": 1})
assert len({e["event_id"] for e in EVENTS}) == len(EVENTS)
assert max(e["day"] for e in EVENTS if e["tenant"] == "tenant-a") == 9
assert any(n == "u11" and created == 7 for t, n, created in NODES if t == "tenant-a")


## 3. 时序快照与候选全集

候选全集 `C_t` 是 cutoff 时已创建的同 tenant 节点对，排除 self-loop 和历史已有边。正样本必须同时满足：它属于 `C_t`，且全事件流中的首次成边时间落在 horizon；已有边的重复交互不是“新边正样本”。剩余候选在**封闭世界评估**中记为 0。注意这不等价于现实为负，只表示观察窗口内未出现。随机删边会让未来结构进入特征，并把时间任务偷换成缺边恢复。


In [ ]:
NODE_META = {n: {"tenant": t, "created_day": created} for t, n, created in NODES}
def build_snapshot(events, auth: AuthContext, cutoff: int) -> nx.Graph:
    auth.require("graph:read")
    g = nx.Graph(tenant=auth.tenant, cutoff=cutoff, graph_version=1, schema_version="temporal-social-v1")
    g.add_nodes_from((n, dict(d)) for n, d in NODE_META.items() if d["tenant"] == auth.tenant and d["created_day"] <= cutoff)
    for e in sorted(events, key=lambda x: (x["day"], x["event_id"])):
        if e["tenant"] == auth.tenant and e["day"] <= cutoff and e["u"] in g and e["v"] in g and not g.has_edge(e["u"], e["v"]):
            g.add_edge(e["u"], e["v"], tenant=e["tenant"], first_seen_day=e["day"], event_id=e["event_id"])
    return g

def candidate_universe(g: nx.Graph) -> list[tuple[str, str]]:
    return [pair(u, v) for u, v in combinations(sorted(g.nodes()), 2) if not g.has_edge(u, v)]

def window_positives(events, snapshot: nx.Graph, start: int, end: int) -> set[tuple[str, str]]:
    if start < snapshot.graph["cutoff"] or end <= start:
        raise ValueError("窗口必须位于快照 cutoff 之后")
    tenant, candidates = snapshot.graph["tenant"], set(candidate_universe(snapshot))
    first_seen = {}
    for e in events:
        if e["tenant"] == tenant and e["u"] in snapshot and e["v"] in snapshot:
            p = pair(e["u"], e["v"]); first_seen[p] = min(e["day"], first_seen.get(p, math.inf))
    return {p for p, day in first_seen.items() if p in candidates and start < day <= end}

train_graph, val_graph, test_graph = (build_snapshot(EVENTS, auth_a, c) for c in (3, 5, 7))
train_pos = window_positives(EVENTS, train_graph, 3, 5)
val_pos = window_positives(EVENTS, val_graph, 5, 7)
test_pos = window_positives(EVENTS, test_graph, 7, 9)
assert (len(train_pos), len(val_pos), len(test_pos)) == (5, 5, 6)
assert train_pos <= set(candidate_universe(train_graph)) and val_pos <= set(candidate_universe(val_graph)) and test_pos <= set(candidate_universe(test_graph))
assert all(not test_graph.has_edge(*p) for p in test_pos)  # test 未来边未进入特征快照
repeat_existing = {"event_id": stable_id("tenant-a", "u00", "u01", 4), "tenant": "tenant-a", "u": "u00", "v": "u01", "day": 4}
assert pair("u00", "u01") not in window_positives(EVENTS + [repeat_existing], train_graph, 3, 5)
assert build_snapshot(EVENTS, AuthContext("tenant-b", frozenset({"graph:read"}), "bob"), 7).number_of_edges() == 1


## 4. 可解释结构特征

对候选 `(u,v)`：Common Neighbors 是邻居交集大小；Jaccard 用交集/并集归一化；Adamic–Adar 以 `1/log(degree(z))` 降低高频共同邻居贡献；Preferential Attachment 是两端度数乘积。它们都是历史拓扑启发式，不是因果证据，也不处理边类型、方向与时间衰减。特征 schema 顺序必须固定。


In [ ]:
def pair_features(g: nx.Graph, u: str, v: str) -> np.ndarray:
    if u not in g or v not in g or u == v or g.has_edge(u, v):
        raise ValueError("特征只接受快照内尚未连接的不同节点")
    nu, nv = set(g.neighbors(u)), set(g.neighbors(v)); common = nu & nv; union = nu | nv
    cn = len(common); jaccard = cn / len(union) if union else 0.0
    aa = sum(1.0 / math.log(g.degree(z)) for z in common if g.degree(z) > 1)
    pa = g.degree(u) * g.degree(v)
    same_component = float(nx.has_path(g, u, v))
    return np.array([cn, jaccard, aa, pa, g.degree(u), g.degree(v), same_component], dtype=float)

def matrix_for(g: nx.Graph, pairs):
    return np.vstack([pair_features(g, *p) for p in pairs])

probe = pair_features(train_graph, *sorted(train_pos)[0])
assert probe.shape == (len(FEATURE_ORDER),) and np.isfinite(probe).all()
assert probe[0] >= 0 and probe[3] >= 0
nx_probe_pair = sorted(train_pos)[0]
nx_jaccard = next(nx.jaccard_coefficient(train_graph, [nx_probe_pair]))[2]
assert abs(nx_jaccard - pair_features(train_graph, *nx_probe_pair)[1]) < 1e-12


## 5. Random、hard 与 mixed negative sampling

枚举全部非边在大图上是 `O(V²)`，训练通常采样。random negative 覆盖广但太容易；hard negative 选择共同邻居多却没在 horizon 成边的候选，训练信号强但更容易包含尚未观察到的真实边。mixed 兼顾覆盖和难度。采样器必须记录随机种子、候选生成规则、正负比和 cutoff。评估则尽量使用接近线上候选分布的完整集合，不能只在平衡采样集报 AUC。


In [ ]:
def sample_pairs(g: nx.Graph, positives: set[tuple[str, str]], strategy: str, ratio: int = 3, seed: int = 23):
    universe = candidate_universe(g); negatives = [p for p in universe if p not in positives]
    count = min(len(negatives), ratio * len(positives)); rng = np.random.default_rng(seed)
    random_order = [negatives[i] for i in rng.permutation(len(negatives))]
    hard_order = sorted(negatives, key=lambda p: (pair_features(g, *p)[:3].sum(), p), reverse=True)
    if strategy == "random": chosen = random_order[:count]
    elif strategy == "hard": chosen = hard_order[:count]
    elif strategy == "mixed":
        chosen = []
        for p in hard_order[: count // 2] + random_order:
            if p not in chosen:
                chosen.append(p)
            if len(chosen) == count:
                break
    else:
        raise ValueError("unknown negative strategy")
    pairs = sorted(positives) + chosen; labels = np.array([1] * len(positives) + [0] * len(chosen))
    return pairs, labels

sample_stats = {}
for strategy in ("random", "hard", "mixed"):
    ps, ys = sample_pairs(train_graph, train_pos, strategy)
    sample_stats[strategy] = {"rows": len(ps), "positive_rate": float(ys.mean()), "negative_cn_mean": float(matrix_for(train_graph, ps[len(train_pos):])[:, 0].mean())}
display(pd.DataFrame(sample_stats).T)
assert all(v["rows"] == 20 for v in sample_stats.values())
assert sample_stats["hard"]["negative_cn_mean"] >= sample_stats["random"]["negative_cn_mean"]


## 6. False negative、closed world 与 open world

训练 horizon 内未成边，只能称“窗口内未观察到”，不能证明永不成边。下面用更晚事件做**事后诊断**：部分 day3 时的封闭世界负例会在 day6–9 变成正例。不能在真实训练时用未来 oracle 把它们过滤掉，否则又引入未来泄漏。可行策略包括更长成熟窗口、排除近期活跃/高风险候选、PU learning、按曝光日志定义负例，以及报告标签成熟度。


In [ ]:
train_closed_negatives = set(candidate_universe(train_graph)) - train_pos
known_later = window_positives(EVENTS, train_graph, 5, 9)
retrospective_false_negatives = train_closed_negatives & known_later
print({"closed_world_negatives": len(train_closed_negatives), "later_became_positive": len(retrospective_false_negatives),
       "examples": sorted(retrospective_false_negatives)[:4]})
assert len(retrospective_false_negatives) > 0
assert retrospective_false_negatives.isdisjoint(train_pos)


## 7. 轻量 ranker：validation 选负采样策略

用标准化 + LogisticRegression 学习可解释结构特征权重。三种策略在相同 train 正样本上训练，用 validation 的完整候选全集 AP 选型；AUC 也报告，但在正例稀少时 AP 更直接反映头部精度。test 只在选型与阈值确定后运行一次。


In [ ]:
def labeled_universe(g: nx.Graph, positives: set[tuple[str, str]]):
    ps = candidate_universe(g); ys = np.array([int(p in positives) for p in ps])
    if len(np.unique(ys)) != 2:
        raise ValueError("评估窗口必须同时包含正负例")
    return ps, ys, matrix_for(g, ps)

val_pairs, val_y, val_X = labeled_universe(val_graph, val_pos)
models, validation_rows = {}, []
for strategy in ("random", "hard", "mixed"):
    train_pairs, train_y = sample_pairs(train_graph, train_pos, strategy)
    model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=500, random_state=23))
    model.fit(matrix_for(train_graph, train_pairs), train_y); models[strategy] = model
    p = model.predict_proba(val_X)[:, 1]
    validation_rows.append({"strategy": strategy, "val_auc": roc_auc_score(val_y, p), "val_ap": average_precision_score(val_y, p)})
validation = pd.DataFrame(validation_rows)
best_strategy = validation.sort_values(["val_ap", "strategy"], ascending=[False, True]).iloc[0]["strategy"]
ranker = models[best_strategy]
ranker_artifact = {"model_version": MODEL_VERSION, "tenant": auth_a.tenant, "train_cutoff": train_graph.graph["cutoff"],
                   "strategy": best_strategy, "feature_order": list(FEATURE_ORDER),
                   "feature_schema_id": FEATURE_SCHEMA_ID, "random_seed": 23}
ranker_artifact["artifact_id"] = fingerprint(ranker_artifact)
def validate_ranker_contract(model, model_artifact: dict) -> bool:
    unsigned = {k: v for k, v in model_artifact.items() if k != "artifact_id"}
    if fingerprint(unsigned) != model_artifact.get("artifact_id"):
        raise ValueError("ranker artifact 内容哈希不匹配")
    expected_schema = fingerprint({"version": "link-features-v1", "feature_order": list(FEATURE_ORDER)})
    if tuple(model_artifact.get("feature_order", ())) != FEATURE_ORDER or model_artifact.get("feature_schema_id") != expected_schema:
        raise ValueError("ranker 特征顺序或 schema 不匹配")
    if getattr(model, "n_features_in_", None) != len(FEATURE_ORDER):
        raise ValueError("模型输入维数与特征合同不匹配")
    return True
display(validation.round(4)); print("selected=", best_strategy)
assert best_strategy in models and validation["val_ap"].between(0, 1).all()
assert validate_ranker_contract(ranker, ranker_artifact)
forged_schema = {**ranker_artifact, "feature_order": list(reversed(FEATURE_ORDER))}
forged_schema["artifact_id"] = fingerprint({k: v for k, v in forged_schema.items() if k != "artifact_id"})
try:
    validate_ranker_contract(ranker, forged_schema)
    raise AssertionError("伪造特征顺序未被拒绝")
except ValueError:
    pass


## 8. AUC/AP 之外：按 source 计算 Hits@K 与 MRR

线上通常为一个 source 排候选，而不是把所有 pair 混在一起。对每个 horizon 内至少有一个正目标的 source，按模型分数排序；`MRR` 取第一个相关目标的倒数名次，`Hits@K` 判断前 K 是否命中。必须固定并列分数的稳定排序。这里对无向边的两个端点都建立 query。


In [ ]:
def grouped_ranking_metrics(g: nx.Graph, positives: set[tuple[str, str]], model, k: int = 3):
    targets = {u: set() for u in g}
    for u, v in positives:
        targets[u].add(v); targets[v].add(u)
    reciprocal_ranks, hits, detail = [], [], {}
    for source, relevant in sorted(targets.items()):
        if not relevant:
            continue
        candidates = [v for v in sorted(g) if v != source and not g.has_edge(source, v)]
        oriented_pairs = [pair(source, v) for v in candidates]
        scores = model.predict_proba(matrix_for(g, oriented_pairs))[:, 1]
        ranked = [v for _, v in sorted(zip(scores, candidates), key=lambda x: (-x[0], x[1]))]
        first = min(ranked.index(v) + 1 for v in relevant if v in ranked)
        reciprocal_ranks.append(1.0 / first); hits.append(float(first <= k)); detail[source] = {"first_rank": first, "degree": g.degree(source)}
    return {"mrr": float(np.mean(reciprocal_ranks)), f"hits@{k}": float(np.mean(hits)), "queries": len(hits), "detail": detail}

test_pairs, test_y, test_X = labeled_universe(test_graph, test_pos)
test_probability = ranker.predict_proba(test_X)[:, 1]
test_global = {"auc": roc_auc_score(test_y, test_probability), "ap": average_precision_score(test_y, test_probability),
               "positive_rate": float(test_y.mean()), "candidates": len(test_y)}
test_rank = grouped_ranking_metrics(test_graph, test_pos, ranker, k=3)
print("global=", {k: round(v, 4) if isinstance(v, float) else v for k, v in test_global.items()})
print("grouped=", {k: v for k, v in test_rank.items() if k != "detail"})
assert 0 <= test_global["auc"] <= 1 and 0 <= test_global["ap"] <= 1
assert 0 <= test_rank["mrr"] <= 1 and 0 <= test_rank["hits@3"] <= 1
assert test_rank["queries"] == len(set(sum(([u, v] for u, v in test_pos), [])))


## 9. Cold start：结构特征全零不是“低风险”

u11 在 test cutoff 刚出现且无边，CN/Jaccard/AA/PA 都几乎无信息。其排名主要由截距和并列规则决定，不能解释成确定的负预测。生产需要内容/画像模型、流行度或业务规则回退，并单独报告 cold-start coverage 与质量。


In [ ]:
cold_nodes = [n for n in test_graph if test_graph.degree(n) == 0]
cold_detail = {n: test_rank["detail"].get(n) for n in cold_nodes}
print("cold_start=", cold_detail)
assert cold_nodes == ["u11"]
assert np.allclose(pair_features(test_graph, "u11", "u00")[:4], 0.0)
assert pair_features(test_graph, "u11", "u00")[4] == 0.0
assert test_rank["detail"]["u11"]["degree"] == 0


## 10. 校准与阈值只能看 validation

排序分数不天然是可校准概率，尤其负采样改变了训练先验。下面在 validation 上报告 Brier score，并以 macro-F1 选择演示阈值；test 只应用这个冻结阈值。生产可用独立校准集做 Platt/isotonic，并按“每天最多联系多少人”这样的容量约束选阈值。小样本校准值不具有统计稳定性。


In [ ]:
val_probability = ranker.predict_proba(val_X)[:, 1]
threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 19):
    threshold_rows.append((float(threshold), f1_score(val_y, val_probability >= threshold, average="macro")))
selected_threshold, val_threshold_f1 = sorted(threshold_rows, key=lambda x: (-x[1], x[0]))[0]
test_threshold_f1 = f1_score(test_y, test_probability >= selected_threshold, average="macro")
calibration_report = {"val_brier": brier_score_loss(val_y, val_probability), "selected_threshold": selected_threshold,
                      "val_macro_f1": val_threshold_f1, "test_macro_f1": test_threshold_f1}
display(calibration_report)
assert 0.05 <= selected_threshold <= 0.95
assert all(0 <= v <= 1 for v in calibration_report.values())


## 11. 在线候选生成：两跳召回 + 流行度回退

`O(V²)` 全量打分不可上线。常见做法是先召回两跳非邻居、同群组、ANN 或业务白名单，再由 ranker 排序。两跳对 cold start 没有候选，因此用 tenant 内流行节点回退。召回阶段决定性能上限，必须报告 future-positive coverage；权限过滤、屏蔽关系和合规规则应在打分前执行。


In [ ]:
def generate_candidates(g: nx.Graph, auth: AuthContext, source: str, model, model_artifact: dict, limit: int = 5):
    auth.require("model:predict")
    validate_ranker_contract(model, model_artifact)
    if g.graph["tenant"] != auth.tenant or model_artifact["tenant"] != auth.tenant or source not in g or NODE_META[source]["tenant"] != auth.tenant:
        raise PermissionError("source 不在授权快照")
    neighbors = set(g.neighbors(source)); two_hop = set()
    for n in neighbors:
        two_hop.update(g.neighbors(n))
    eligible = lambda v: v != source and v not in neighbors and NODE_META[v]["tenant"] == auth.tenant
    selected = sorted((v for v in two_hop if eligible(v)), key=lambda v: (-g.degree(v), v))
    route = "two_hop"
    if len(selected) < limit:
        route += "+popularity_fallback"
        for v in sorted((v for v in g if eligible(v)), key=lambda v: (-g.degree(v), v)):
            if v not in selected:
                selected.append(v)
            if len(selected) >= limit:
                break
    pairs = [pair(source, v) for v in selected[:limit]]
    scores = model.predict_proba(matrix_for(g, pairs))[:, 1] if pairs else np.array([])
    ranked = sorted(zip(selected[:limit], scores), key=lambda x: (-x[1], x[0]))
    return ranked, {"route": route, "cutoff": g.graph["cutoff"], "source_degree": g.degree(source), "retrieved": len(ranked),
                    "model_version": model_artifact["model_version"], "artifact_id": model_artifact["artifact_id"],
                    "feature_schema_id": model_artifact["feature_schema_id"]}

online_results, online_trace = generate_candidates(test_graph, auth_a, "u11", ranker, ranker_artifact, limit=5)
assert online_trace["source_degree"] == 0 and "fallback" in online_trace["route"]
assert len(online_results) == 5 and all(NODE_META[v]["tenant"] == "tenant-a" for v, _ in online_results)
assert online_trace["model_version"] == MODEL_VERSION and online_trace["artifact_id"] == ranker_artifact["artifact_id"]


## 12. 增量图：幂等与迟到事件

在线快照不能每次从头重建。增量状态需要 event-status 集合、watermark、tenant 分区和审计；`applied/late/existing_edge` 等所有终态都必须幂等。正常事件要以同一事务原子提交边、cutoff、watermark 和 graph version；早于 watermark 的未知迟到事件进入旁路重算队列，不能静默改写正在服务的快照。下面用 copy-on-write 演示原子可见性，生产应替换为数据库事务或 compare-and-swap。删除/撤销还需独立事件类型与版本化回放。


In [ ]:
class IncrementalGraph:
    def __init__(self, graph: nx.Graph):
        self.graph = graph.copy(); self.watermark = graph.graph["cutoff"]; self.version = int(graph.graph["graph_version"])
        self.event_status = {d["event_id"]: "applied" for _, _, d in graph.edges(data=True)}
        self.late_queue = []; self.audit = []
    def apply(self, auth: AuthContext, event: dict):
        auth.require("graph:write")
        if event["tenant"] != auth.tenant or self.graph.graph["tenant"] != auth.tenant:
            raise PermissionError("tenant mismatch")
        if event["event_id"] in self.event_status:
            return "duplicate"
        if event["day"] < self.watermark:
            self.event_status[event["event_id"]] = "late"; self.late_queue.append(dict(event))
            self.audit.append((self.version, event["event_id"], auth.principal, "LATE")); return "late"
        if event["u"] not in self.graph or event["v"] not in self.graph:
            raise KeyError("端点尚未进入快照")
        if event["u"] == event["v"]:
            raise ValueError("不允许 self-loop event")
        if self.graph.has_edge(event["u"], event["v"]):
            self.event_status[event["event_id"]] = "existing_edge"
            self.audit.append((self.version, event["event_id"], auth.principal, "EXISTING_EDGE")); return "existing_edge"
        new_watermark, new_version = max(self.watermark, event["day"]), self.version + 1
        new_graph = self.graph.copy()
        new_graph.add_edge(event["u"], event["v"], tenant=auth.tenant, first_seen_day=event["day"], event_id=event["event_id"])
        new_graph.graph["cutoff"] = new_watermark; new_graph.graph["graph_version"] = new_version
        self.graph, self.watermark, self.version = new_graph, new_watermark, new_version
        self.event_status[event["event_id"]] = "applied"
        self.audit.append((self.version, event["event_id"], auth.principal, "APPLIED")); return "applied"

state = IncrementalGraph(test_graph); next_event = next(e for e in EVENTS if e["tenant"] == "tenant-a" and e["day"] == 8)
assert state.apply(auth_a, next_event) == "applied"
assert state.apply(auth_a, next_event) == "duplicate" and state.version == 2
assert state.graph.graph["cutoff"] == state.watermark == 8 and state.graph.graph["graph_version"] == state.version
assert max(d["first_seen_day"] for _, _, d in state.graph.edges(data=True)) <= state.graph.graph["cutoff"]
late = {"event_id": stable_id("late-demo"), "tenant": "tenant-a", "u": "u02", "v": "u06", "day": 2}
assert state.apply(auth_a, late) == "late" and len(state.late_queue) == 1
late_version = state.version
assert state.apply(auth_a, late) == "duplicate" and len(state.late_queue) == 1 and state.version == late_version


## 13. 可观测性、告警与生产替换点

离线：每个窗口的快照水位、V/E、候选数、正例率、负采样策略/种子、false-negative 成熟度、AUC/AP/Hits/MRR、cold-start 切片和置信区间。在线：候选召回量/coverage、fallback 率、特征缺失、分数与度分布漂移、P50/P95/P99、watermark lag、late/duplicate、越权拒绝和人工反馈。不要把 user ID 作为高基数监控标签。

生产替换点：图库/流处理维护时态邻接，图数据库或 KV 邻居服务做召回，GBDT/深度双塔/GNN 做 ranker，专门模型注册与校准服务管理版本。若图服务超时，可回退到经权限过滤的流行度/内容召回，并在响应 trace 标注 degraded。


In [ ]:
def observability_record(g: nx.Graph, positives, probabilities, labels, latency_ms: float):
    return {"tenant": g.graph["tenant"], "cutoff": g.graph["cutoff"], "nodes": len(g), "edges": g.number_of_edges(),
            "candidates": len(labels), "positive_rate": float(np.mean(labels)), "score_mean": float(np.mean(probabilities)),
            "cold_start_rate": float(np.mean([g.degree(n) == 0 for n in g])), "latency_ms": latency_ms,
            "model_strategy": best_strategy, "model_version": ranker_artifact["model_version"],
            "artifact_id": ranker_artifact["artifact_id"], "feature_schema": FEATURE_SCHEMA_ID}

started = time.perf_counter(); _ = ranker.predict_proba(test_X)[:, 1]; latency_ms = (time.perf_counter() - started) * 1000
telemetry = observability_record(test_graph, test_pos, test_probability, test_y, latency_ms)
display(telemetry)
assert telemetry["candidates"] == len(candidate_universe(test_graph))
assert telemetry["tenant"] == "tenant-a" and telemetry["latency_ms"] >= 0
assert 0 < telemetry["positive_rate"] < 1


## 14. 常见失败与上线清单

- 随机拆现有边：未来泄漏；改为 event-time split。
- 在全图先算 CN/度数：test 边进入特征；每个 cutoff 单独建快照。
- 把随机非边当确定负例：忽略开放世界与曝光；记录标签成熟度/使用 PU 思路。
- 只在 1:1 采样集报 accuracy/AUC：分布不真实；在实际候选集报告 AP 和按 query 排名。
- 全量 `V²` 在线打分：不可扩展；召回后 rank。
- test 选阈值或 K：评估污染；validation 冻结。
- cold start 当低概率：把缺信息误作负证据；单独回退与切片。
- 先打分后 ACL：泄漏；tenant/屏蔽/合规在候选前过滤。


In [ ]:
# 汇总合同测试
assert train_graph.graph["cutoff"] < val_graph.graph["cutoff"] < test_graph.graph["cutoff"]
assert set(train_graph).issubset(val_graph) and set(val_graph).issubset(test_graph)
assert all(NODE_META[n]["tenant"] == auth_a.tenant for n in test_graph)
assert set(FEATURE_ORDER) == {"common_neighbors", "jaccard", "adamic_adar", "preferential_attachment", "degree_u", "degree_v", "same_component"}
assert best_strategy in {"random", "hard", "mixed"} and state.watermark == state.graph.graph["cutoff"] == 8
assert validate_ranker_contract(ranker, ranker_artifact) and online_trace["artifact_id"] == ranker_artifact["artifact_id"]
print("时序链接预测合同测试通过；累计 assert > 30。")


## 15. 原始与官方资料

- Liben-Nowell & Kleinberg, *The Link Prediction Problem for Social Networks*：https://people.csail.mit.edu/dln/papers/link/paper.pdf
- NetworkX 官方 Link Prediction API（Jaccard、Adamic–Adar、Preferential Attachment）：https://networkx.org/documentation/stable/reference/algorithms/link_prediction.html
- Mikolov et al., *Distributed Representations of Words and Phrases and their Compositionality*（负采样思想的重要来源）：https://proceedings.neurips.cc/paper/2013/hash/9aa42b31882ec039965f3c4923ce901b-Abstract.html
- Hamilton, Ying & Leskovec, *Inductive Representation Learning on Large Graphs*：https://proceedings.neurips.cc/paper/2017/hash/5dd9db5e033da9c6fb5ba83c7a7ebea9-Abstract.html
- Poursafaei et al., *Towards Better Evaluation for Dynamic Link Prediction*（时间评估与困难负例）：https://proceedings.neurips.cc/paper_files/paper/2022/hash/d49042a5d49818711c401d34172f9900-Abstract-Datasets_and_Benchmarks.html
- Huang et al., *Temporal Graph Benchmark*（固定时序切分、按 source 排名与负采样协议）：https://proceedings.neurips.cc/paper_files/paper/2023/hash/066b98e63313162f6562b35962671288-Abstract-Datasets_and_Benchmarks.html

这些来源给出经典结构指标、表示学习背景，以及直接面向动态链接预测的时序切分、困难负例和按 source 排名协议；candidate universe、权限、watermark、模型阈值、版本绑定和可观测合同是本 Notebook 的工程重点。
